**Team Member C — Model Validator**

Parameters used throughout (unless a question states otherwise):
- S0 = 80, r = 5.5%, sigma = 35%, T = 3 months
- Heston: v0 = 3.2%, kappa = 1.85, theta = 0.045 (sigma above used as the vol-of-vol parameter)
- Merton: jump mean mu = -0.5, jump std delta = 0.22 (sigma above used as the diffusion volatility)


In [1]:
import numpy as np


# Global parameters (per GWP2 instructions)

S0    = 80.0
r     = 0.055
sigma = 0.35      # general "sigma" -> Heston vol-of-vol (xi) AND Merton diffusion vol
T     = 0.25      # 3 months

# Heston-specific
v0, kappa, theta = 0.032, 1.85, 0.045

# Merton-specific
muJ, deltaJ = -0.5, 0.22

N_STEPS = 63       # ~ trading days in 3 months (daily monitoring, good for barriers)
N_PATHS = 60000    # antithetic -> 120,000 effective paths

rng_master = np.random.default_rng(20250601)


# HESTON PATH SIMULATOR (Full Truncation Euler)

def heston_paths(rho, n_paths=N_PATHS, n_steps=N_STEPS, seed=None, S0_=S0):
    rng = np.random.default_rng(seed) if seed is not None else rng_master
    dt = T / n_steps
    half = n_paths
    Z1 = rng.standard_normal((half, n_steps))
    Z2 = rng.standard_normal((half, n_steps))
    # antithetic
    Z1 = np.vstack([Z1, -Z1])
    Z2 = np.vstack([Z2, -Z2])
    Zv = Z1
    Zs = rho * Z1 + np.sqrt(1 - rho**2) * Z2

    n_tot = 2 * half
    S = np.empty((n_tot, n_steps + 1))
    v = np.empty((n_tot, n_steps + 1))
    S[:, 0] = S0_
    v[:, 0] = v0

    for t in range(n_steps):
        vt = np.maximum(v[:, t], 0.0)         # full truncation
        sqrt_vt = np.sqrt(vt)
        v[:, t+1] = v[:, t] + kappa*(theta - vt)*dt + sigma*sqrt_vt*np.sqrt(dt)*Zv[:, t]
        S[:, t+1] = S[:, t] * np.exp((r - 0.5*vt)*dt + sqrt_vt*np.sqrt(dt)*Zs[:, t])
    return S  # shape (n_tot, n_steps+1)

def heston_price(K, rho, option='call', n_paths=N_PATHS, n_steps=N_STEPS, seed=1, S0_=S0):
    S = heston_paths(rho, n_paths, n_steps, seed=seed, S0_=S0_)
    ST = S[:, -1]
    if option == 'call':
        payoff = np.maximum(ST - K, 0.0)
    else:
        payoff = np.maximum(K - ST, 0.0)
    price = np.exp(-r*T) * payoff.mean()
    se = np.exp(-r*T) * payoff.std(ddof=1) / np.sqrt(len(payoff))
    return price, se

def heston_call_put(K, rho, n_paths=N_PATHS, n_steps=N_STEPS, seed=1, S0_=S0):
    """Same simulated paths -> consistent call & put (clean parity check)."""
    S = heston_paths(rho, n_paths, n_steps, seed=seed, S0_=S0_)
    ST = S[:, -1]
    call = np.exp(-r*T) * np.maximum(ST - K, 0.0).mean()
    put  = np.exp(-r*T) * np.maximum(K - ST, 0.0).mean()
    return call, put


# MERTON JUMP-DIFFUSION PATH SIMULATOR

def merton_paths(lam, n_paths=N_PATHS, n_steps=N_STEPS, seed=None, S0_=S0):
    rng = np.random.default_rng(seed) if seed is not None else rng_master
    dt = T / n_steps
    k = np.exp(muJ + 0.5*deltaJ**2) - 1.0      # compensator E[Y-1]
    half = n_paths
    Zs = rng.standard_normal((half, n_steps))
    Zs = np.vstack([Zs, -Zs])                  # antithetic for diffusion part
    n_tot = 2*half
    Npois = rng.poisson(lam*dt, size=(n_tot, n_steps))
    # jump sizes: sum of Npois iid N(muJ, deltaJ^2) draws per step, vectorized via normal approx
    # exact: sample per-step aggregate jump as Normal(Npois*muJ, Npois*deltaJ^2) (valid since sum of normals)
    jump_mean = Npois * muJ
    jump_var  = Npois * deltaJ**2
    J = rng.standard_normal((n_tot, n_steps)) * np.sqrt(np.maximum(jump_var, 0)) + jump_mean

    S = np.empty((n_tot, n_steps+1))
    S[:, 0] = S0_
    drift = (r - lam*k - 0.5*sigma**2) * dt
    for t in range(n_steps):
        S[:, t+1] = S[:, t] * np.exp(drift + sigma*np.sqrt(dt)*Zs[:, t] + J[:, t])
    return S

def merton_call_put(K, lam, n_paths=N_PATHS, n_steps=N_STEPS, seed=2, S0_=S0):
    S = merton_paths(lam, n_paths, n_steps, seed=seed, S0_=S0_)
    ST = S[:, -1]
    call = np.exp(-r*T) * np.maximum(ST - K, 0.0).mean()
    put  = np.exp(-r*T) * np.maximum(K - ST, 0.0).mean()
    return call, put


# Merton closed-form (Merton 1976) -- used only as an independent cross-check

from scipy.stats import norm
from math import factorial, exp, log, sqrt

def bs_price(S0_, K, r_, sigma_, T_, option='call'):
    d1 = (log(S0_/K) + (r_ + 0.5*sigma_**2)*T_) / (sigma_*sqrt(T_))
    d2 = d1 - sigma_*sqrt(T_)
    if option == 'call':
        return S0_*norm.cdf(d1) - K*exp(-r_*T_)*norm.cdf(d2)
    else:
        return K*exp(-r_*T_)*norm.cdf(-d2) - S0_*norm.cdf(-d1)

def merton_closed_form(K, lam, option='call', n_terms=60):
    k = exp(muJ + 0.5*deltaJ**2) - 1.0
    lam_p = lam*(1+k)
    total = 0.0
    for n in range(n_terms):
        r_n = r - lam*k + n*log(1+k)/T
        sigma_n = sqrt(sigma**2 + n*deltaJ**2/T)
        weight = exp(-lam_p*T) * (lam_p*T)**n / factorial(n)
        total += weight * bs_price(S0, K, r_n, sigma_n, T, option)
    return total


# DELTA / GAMMA by finite difference, common random numbers (same seed
# across the bumped/unbumped runs to cancel Monte-Carlo noise)

H_BUMP = 0.8   # 1% of S0=80

def heston_delta_gamma(K, rho, option='call', h=H_BUMP, seed=777):
    p_up, _   = heston_price(K, rho, option, seed=seed, S0_=S0+h)
    p_mid, _  = heston_price(K, rho, option, seed=seed, S0_=S0)
    p_dn, _   = heston_price(K, rho, option, seed=seed, S0_=S0-h)
    delta = (p_up - p_dn) / (2*h)
    gamma = (p_up - 2*p_mid + p_dn) / (h**2)
    return delta, gamma, p_mid

def merton_delta_gamma(K, lam, option='call', h=H_BUMP, seed=888):
    Su = merton_paths(lam, seed=seed, S0_=S0+h)[:, -1]
    Sm = merton_paths(lam, seed=seed, S0_=S0)[:, -1]
    Sd = merton_paths(lam, seed=seed, S0_=S0-h)[:, -1]
    if option == 'call':
        p_up = exp(-r*T)*np.maximum(Su-K,0).mean()
        p_mid= exp(-r*T)*np.maximum(Sm-K,0).mean()
        p_dn = exp(-r*T)*np.maximum(Sd-K,0).mean()
    else:
        p_up = exp(-r*T)*np.maximum(K-Su,0).mean()
        p_mid= exp(-r*T)*np.maximum(K-Sm,0).mean()
        p_dn = exp(-r*T)*np.maximum(K-Sd,0).mean()
    delta = (p_up - p_dn) / (2*h)
    gamma = (p_up - 2*p_mid + p_dn) / (h**2)
    return delta, gamma, p_mid


# AMERICAN CALL under Heston, via Longstaff-Schwartz Monte Carlo (LSM)

def heston_american_call_lsm(K, rho, n_paths=N_PATHS, n_steps=N_STEPS, seed=555, S0_=S0):
    S = heston_paths(rho, n_paths, n_steps, seed=seed, S0_=S0_)
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    half = n_paths
    Z1 = rng.standard_normal((half, n_steps)); Z2 = rng.standard_normal((half, n_steps))
    Z1 = np.vstack([Z1, -Z1]); Z2 = np.vstack([Z2, -Z2])
    Zv = Z1
    n_tot = 2*half
    v = np.empty((n_tot, n_steps+1)); v[:,0] = v0
    for t in range(n_steps):
        vt = np.maximum(v[:,t], 0.0)
        v[:,t+1] = v[:,t] + kappa*(theta-vt)*dt + sigma*np.sqrt(vt)*np.sqrt(dt)*Zv[:,t]

    disc = exp(-r*dt)
    cashflow = np.maximum(S[:, -1] - K, 0.0)
    exercise_time = np.full(n_tot, n_steps)

    for t in range(n_steps-1, 0, -1):
        St = S[:, t]
        itm = St > K
        if itm.sum() < 5:
            continue
        X = St[itm]; V = v[itm, t]
        Y = cashflow[itm] * disc**(exercise_time[itm]-t)
        basis = np.column_stack([np.ones_like(X), X, X**2, V, X*V])
        coef, *_ = np.linalg.lstsq(basis, Y, rcond=None)
        cont_est = basis @ coef
        intrinsic = X - K
        exercise_now = intrinsic > cont_est
        idx = np.where(itm)[0][exercise_now]
        cashflow[idx] = intrinsic[exercise_now]
        exercise_time[idx] = t

    disc_cf = cashflow * disc**exercise_time
    price = disc_cf.mean()
    se = disc_cf.std(ddof=1)/np.sqrt(n_tot)
    return price, se


# BARRIER OPTIONS (discrete monitoring on the simulated grid)

def heston_up_in_call(K, barrier, rho, n_paths=N_PATHS, n_steps=N_STEPS, seed=666, S0_=S0):
    S = heston_paths(rho, n_paths, n_steps, seed=seed, S0_=S0_)
    hit = (S.max(axis=1) >= barrier)
    payoff = np.where(hit, np.maximum(S[:, -1]-K, 0.0), 0.0)
    price = exp(-r*T)*payoff.mean()
    se = exp(-r*T)*payoff.std(ddof=1)/np.sqrt(len(payoff))
    vanilla = exp(-r*T)*np.maximum(S[:,-1]-K,0).mean()
    return price, se, vanilla

def merton_down_in_put(K, barrier, lam, n_paths=N_PATHS, n_steps=N_STEPS, seed=668, S0_=S0):
    S = merton_paths(lam, n_paths, n_steps, seed=seed, S0_=S0_)
    hit = (S.min(axis=1) <= barrier)
    payoff = np.where(hit, np.maximum(K-S[:, -1], 0.0), 0.0)
    price = exp(-r*T)*payoff.mean()
    se = exp(-r*T)*payoff.std(ddof=1)/np.sqrt(len(payoff))
    vanilla = exp(-r*T)*np.maximum(K-S[:,-1],0).mean()
    return price, se, vanilla


## Question 5 — Heston Model, ATM European Call & Put (rho = -0.30)
Priced via Monte-Carlo simulation of the Heston stochastic-volatility process.

In [2]:
c5, p5 = heston_call_put(80, -0.30, seed=11)
print(f"European Call (Heston, rho=-0.30): ${c5:.2f}")
print(f"European Put  (Heston, rho=-0.30): ${p5:.2f}")


European Call (Heston, rho=-0.30): $3.47
European Put  (Heston, rho=-0.30): $2.38


## Question 6 — Heston Model, ATM European Call & Put (rho = -0.70)

In [3]:
c6, p6 = heston_call_put(80, -0.70, seed=12)
print(f"European Call (Heston, rho=-0.70): ${c6:.2f}")
print(f"European Put  (Heston, rho=-0.70): ${p6:.2f}")


European Call (Heston, rho=-0.70): $3.50
European Put  (Heston, rho=-0.70): $2.41


## Question 7 — Delta & Gamma for the Q5 / Q6 options
Computed by bumping S0 up and down by 1% (h=$0.80) and re-pricing with the common random numbers, which cancels most Monte-Carlo noise in the finite-difference estimate.

In [4]:
d5c,g5c,_ = heston_delta_gamma(80,-0.30,'call', seed=101)
d5p,g5p,_ = heston_delta_gamma(80,-0.30,'put',  seed=101)
d6c,g6c,_ = heston_delta_gamma(80,-0.70,'call', seed=102)
d6p,g6p,_ = heston_delta_gamma(80,-0.70,'put',  seed=102)

import pandas as pd
q7_table = pd.DataFrame({
    'Option':['Call (rho=-0.30)','Put (rho=-0.30)','Call (rho=-0.70)','Put (rho=-0.70)'],
    'Delta':[round(d5c,4), round(d5p,4), round(d6c,4), round(d6p,4)],
    'Gamma':[round(g5c,5), round(g5p,5), round(g6c,5), round(g6p,5)],
})
q7_table


,Option,Delta,Gamma
0,Call (rho=-0.30),0.6050,0.05526
1,Put (rho=-0.30),-0.3951,0.05526
2,Call (rho=-0.70),0.6333,0.05083
3,Put (rho=-0.70),-0.3667,0.05083


## Question 8 — Merton Model, ATM European Call & Put (jump intensity lambda = 0.75)
Priced via Monte-Carlo simulation of the Merton jump-diffusion process (compound Poisson jumps + GBM diffusion). Cross-validated against the closed-form Merton (1976) series solution.

In [5]:
c8, p8 = merton_call_put(80, 0.75, seed=21)
print(f"European Call (Merton, lambda=0.75): ${c8:.2f}")
print(f"European Put  (Merton, lambda=0.75): ${p8:.2f}")
print(f"  [Closed-form cross-check -> Call: ${merton_closed_form(80,0.75,'call'):.2f}, Put: ${merton_closed_form(80,0.75,'put'):.2f}]")


European Call (Merton, lambda=0.75): $8.30
European Put  (Merton, lambda=0.75): $7.17
  [Closed-form cross-check -> Call: $8.31, Put: $7.21]


## Question 9 — Merton Model, ATM European Call & Put (jump intensity lambda = 0.25)

In [6]:
c9, p9 = merton_call_put(80, 0.25, seed=22)
print(f"European Call (Merton, lambda=0.25): ${c9:.2f}")
print(f"European Put  (Merton, lambda=0.25): ${p9:.2f}")
print(f"  [Closed-form cross-check -> Call: ${merton_closed_form(80,0.25,'call'):.2f}, Put: ${merton_closed_form(80,0.25,'put'):.2f}]")


European Call (Merton, lambda=0.25): $6.84
European Put  (Merton, lambda=0.25): $5.73
  [Closed-form cross-check -> Call: $6.83, Put: $5.73]


## Question 10 — Delta & Gamma for the Q8 / Q9 options

In [7]:
d8c,g8c,_ = merton_delta_gamma(80,0.75,'call', seed=201)
d8p,g8p,_ = merton_delta_gamma(80,0.75,'put',  seed=201)
d9c,g9c,_ = merton_delta_gamma(80,0.25,'call', seed=202)
d9p,g9p,_ = merton_delta_gamma(80,0.25,'put',  seed=202)

q10_table = pd.DataFrame({
    'Option':['Call (lambda=0.75)','Put (lambda=0.75)','Call (lambda=0.25)','Put (lambda=0.25)'],
    'Delta':[round(d8c,4), round(d8p,4), round(d9c,4), round(d9p,4)],
    'Gamma':[round(g8c,5), round(g8p,5), round(g9c,5), round(g9p,5)],
})
q10_table


,Option,Delta,Gamma
0,Call (lambda=0.75),0.6491,0.02230
1,Put (lambda=0.75),-0.3517,0.02230
2,Call (lambda=0.25),0.5975,0.02652
3,Put (lambda=0.25),-0.4023,0.02652


## Question 11 (Team Member C) — Put-Call Parity Check
Theoretical European parity: **C - P = S0 - K e^(-rT)**.

This holds regardless of the volatility model used (Heston, Merton, or Black-Scholes), because it is derived purely from a no-arbitrage replication argument, not from any assumption about the volatility process. We check it against the Monte-Carlo prices from Q5, Q6, Q8 and Q9.

In [8]:
K = 80.0
theo_rhs = S0 - K*exp(-r*T)

parity_check = pd.DataFrame({
    'Scenario':['Q5: Heston rho=-0.30','Q6: Heston rho=-0.70','Q8: Merton lambda=0.75','Q9: Merton lambda=0.25'],
    'Call':[round(c5,2), round(c6,2), round(c8,2), round(c9,2)],
    'Put':[round(p5,2), round(p6,2), round(p8,2), round(p9,2)],
    'C - P (simulated)':[round(c5-p5,4), round(c6-p6,4), round(c8-p8,4), round(c9-p9,4)],
    'S0 - K*e^(-rT) (theory)':[round(theo_rhs,4)]*4,
})
parity_check['Difference'] = parity_check['C - P (simulated)'] - parity_check['S0 - K*e^(-rT) (theory)']
parity_check


,Scenario,Call,Put,C - P (simulated),S0 - K*e^(-rT) (theory),Difference
0,Q5: Heston rho=-0.30,3.47,2.38,1.0886,1.0925,-0.0039
1,Q6: Heston rho=-0.70,3.50,2.41,1.0898,1.0925,-0.0027
2,Q8: Merton lambda=0.75,8.30,7.17,1.1239,1.0925,0.0314
3,Q9: Merton lambda=0.25,6.84,5.73,1.1114,1.0925,0.0189


All four differences are under 1% of the option price, which is fully attributable to Monte-Carlo sampling noise rather than a model or implementation error. Put-call parity holds for every Heston and Merton scenario tested. The Q5/Q6/Q8/Q9 prices are internally consistent and arbitrage-free.

## Question 12: 7-Strike Grid (Heston vs Merton)
Strikes chosen from approximate moneyness (stock/strike) = 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, giving 3 OTM calls, 1 ATM call, and 3 ITM calls. We use the base-case parameters from Q5 (Heston, rho=-0.30) and Q8 (Merton, lambda=0.75).

In [9]:
moneyness = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
strikes = [round(S0/m, 2) for m in moneyness]

heston_calls, merton_calls = [], []
for K_ in strikes:
    c,_ = heston_price(K_, -0.30, 'call', seed=303)
    heston_calls.append(round(c,2))
for K_ in strikes:
    S_ = merton_paths(0.75, seed=404)[:, -1]
    merton_calls.append(round(exp(-r*T)*np.maximum(S_-K_,0).mean(), 2))

q12_table = pd.DataFrame({
    'Moneyness (S/K)': moneyness,
    'Strike': strikes,
    'Type': ['Deep OTM','OTM','OTM','ATM','ITM','ITM','Deep ITM'],
    'Heston Call': heston_calls,
    'Merton Call': merton_calls,
})
q12_table


,Moneyness (S/K),Strike,Type,Heston Call,Merton Call
0,0.85,94.12,Deep OTM,0.14,2.84
1,0.90,88.89,OTM,0.56,4.39
2,0.95,84.21,OTM,1.61,6.25
3,1.00,80.00,ATM,3.48,8.34
4,1.05,76.19,ITM,5.98,10.57
5,1.10,72.73,ITM,8.77,12.86
6,1.15,69.57,Deep ITM,11.60,15.13


For both models, call prices fall monotonically as the strike rises, moneyness increases past 1. Moneyness = S/K, so moneyness > 1 means K < S0, i.e. ITM. This is exactly the behavior basic option theory predicts, a call becomes less valuable as its strike rises, regardless of which volatility model is used underneath. The Merton-model prices sit above the Heston-model prices at every strike, reflecting the extra tail risk that downward price jumps add to the distribution of S(T).